# Phase 4: Baseline Model Training

**Predicting 30-Day Hospital Readmission for Diabetic Patients**

Train 4 baseline classifiers (LR, RF, XGBoost, SVM) using 5-fold stratified cross-validation, compare class imbalance strategies, and identify which models warrant tuning in Phase 5.

**Inputs:** Phase 3 artifacts (X_train, X_test, y_train, y_test, preprocessor, feature lists)
**Outputs:** CV results, baseline models, imbalance comparison -> Phase 5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, pickle, warnings, time, os
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

COLORS = {
    'primary': '#1B2A4A', 'steel': '#4A7FB5', 'teal': '#2AA198',
    'orange': '#E67E22', 'red': '#E74C3C', 'green': '#2ECC71', 'gold': '#D4A017'
}

ARTIFACTS_DIR = '../artifacts'
os.makedirs('../figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Environment ready.')

### Load Phase 3 Artifacts

In [ ]:
# Load Phase 3 artifacts
X_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_train.parquet')
X_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_test.parquet')
y_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_train.parquet').squeeze()
y_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_test.parquet').squeeze()
preprocessor = joblib.load(f'{ARTIFACTS_DIR}/phase3_preprocessor.joblib')

with open(f'{ARTIFACTS_DIR}/phase3_meta.pkl', 'rb') as f:
    meta = pickle.load(f)

numeric_features     = meta['numeric_features']
categorical_features = meta['categorical_features']
binary_features      = meta['binary_features']
all_features         = meta['all_features']
target               = meta['target']

# Refit preprocessor on X_train
X_train_transformed = preprocessor.fit_transform(X_train)
feature_names = preprocessor.get_feature_names_out()

print(f'Phase 3 artifacts loaded')
print(f'  Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'  Train positive rate: {y_train.mean()*100:.1f}%')
print(f'  Encoded feature count: {X_train_transformed.shape[1]}')

### 4.0 Utility Functions for Modeling Phases

Based on peer review feedback, define reusable functions for processes that repeat across Phases 4-6: model evaluation, threshold-aware prediction, and business impact simulation. Each function is called multiple times -- during cross-validation comparison, final test-set evaluation, and cost-benefit analysis. Wrapping them here reduces code duplication and ensures consistent metric reporting.

In [ ]:
# === Utility Functions for Modeling Phases ===
# Defined once here, reused across Phases 4, 5, and 6.

from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)


def evaluate_model(y_true, y_pred, y_prob, model_name="Model", show_matrix=True):
    """Compute and print standard classification metrics.

    Args:
        y_true: actual labels
        y_pred: predicted labels (binary)
        y_prob: predicted probabilities for positive class
        model_name: display name for printing
        show_matrix: whether to plot confusion matrix

    Returns:
        dict of metric values
    """
    metrics = {
        'auc': roc_auc_score(y_true, y_prob),
        'recall': recall_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred),
    }

    print(f"\n{'=' * 50}")
    print(f"{model_name} -- Evaluation Metrics")
    print(f"{'=' * 50}")
    print(f"  ROC-AUC:   {metrics['auc']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Not Readmit', 'Readmit <30'])}")

    if show_matrix:
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred,
            display_labels=['Not Readmit', 'Readmit <30'],
            cmap='Blues', ax=ax, values_format=','
        )
        ax.set_title(f'{model_name} -- Confusion Matrix', fontweight='bold')
        plt.tight_layout()
        plt.show()

    return metrics


def predict_at_threshold(y_prob, threshold=0.5):
    """Convert probabilities to binary predictions at a custom threshold.

    Args:
        y_prob: predicted probabilities for positive class
        threshold: classification cutoff (default 0.5)

    Returns:
        numpy array of binary predictions
    """
    return (y_prob >= threshold).astype(int)


def compute_business_impact(y_true, y_prob, threshold,
                            penalty_saved_per_tp=13000,
                            intervention_cost_per_flag=500):
    """Compute cost-adjusted net benefit at a given threshold.

    Simulates the financial impact of deploying the model at a hospital
    with the given cost assumptions.

    Args:
        y_true: actual labels
        y_prob: predicted probabilities
        threshold: classification cutoff
        penalty_saved_per_tp: estimated $ saved per correctly flagged readmission
        intervention_cost_per_flag: $ cost per patient flagged (TP + FP)

    Returns:
        dict with TP, FP, FN, net_benefit, and per-patient metrics
    """
    y_pred = predict_at_threshold(y_prob, threshold)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    total_flagged = tp + fp
    savings = tp * penalty_saved_per_tp
    costs = total_flagged * intervention_cost_per_flag
    net_benefit = savings - costs

    return {
        'threshold': threshold,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'total_flagged': total_flagged,
        'recall': tp / max(tp + fn, 1),
        'precision': tp / max(tp + fp, 1),
        'savings': savings,
        'costs': costs,
        'net_benefit': net_benefit,
    }


print("Utility functions defined:")
print("  evaluate_model()          -- metrics + classification report + optional confusion matrix")
print("  predict_at_threshold()    -- probability to binary at custom cutoff")
print("  compute_business_impact() -- cost-adjusted net benefit simulation")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Calculate scale_pos_weight for XGBoost
# Derive class counts from y_train (already in memory after Phase 3 split).
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_weight = neg_count / max(pos_count, 1)

models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
            solver='saga',
            penalty='l2'
        ))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_leaf=50,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=scale_weight,
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        ))
    ]),
    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(
            kernel='rbf',
            class_weight='balanced',
            probability=True,
            random_state=42
        ))
    ]),
}

print(f"Models defined: {list(models.keys())}")
print(f"Class balance: {neg_count:,} negative, {pos_count:,} positive")
print(f"scale_pos_weight for XGBoost: {scale_weight:.2f}")
print(f"All models use class_weight='balanced' or equivalent.")

### Why These Four Models?

- **Logistic Regression** -- Interpretable baseline whose coefficients map directly to odds ratios. L2 regularization handles correlated features that arise after one-hot encoding. The `saga` solver scales well to the ~56K training set.
- **Random Forest** -- Captures non-linear feature interactions without manual feature crosses. Built-in feature importance ranking, robust to outliers, and `class_weight='balanced'` adjusts the per-tree bootstrap sampling.
- **XGBoost** -- State-of-the-art for tabular classification. The published benchmark for this dataset used gradient boosting with AUC ~0.667 (Strack et al., 2014). `scale_pos_weight` handles class imbalance natively without resampling.
- **SVM (RBF kernel)** -- A margin-based learning paradigm that adds diversity to the comparison. Effective in high-dimensional spaces after one-hot encoding. `probability=True` enables AUC scoring via Platt scaling but increases training time significantly.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import time

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'recall', 'precision', 'f1']

results = {}
for name, pipeline in models.items():
    print(f"\nTraining {name}...")
    if name == 'SVM':
        print("  (SVM with RBF + probability=True may take 10-30 minutes on this dataset size)")
    start = time.time()

    cv_results = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1,
        return_train_score=False
    )

    elapsed = time.time() - start
    results[name] = {
        'auc_mean': cv_results['test_roc_auc'].mean(),
        'auc_std': cv_results['test_roc_auc'].std(),
        'recall_mean': cv_results['test_recall'].mean(),
        'recall_std': cv_results['test_recall'].std(),
        'precision_mean': cv_results['test_precision'].mean(),
        'precision_std': cv_results['test_precision'].std(),
        'f1_mean': cv_results['test_f1'].mean(),
        'f1_std': cv_results['test_f1'].std(),
        'time': elapsed,
        'cv_results': cv_results,
    }

    print(f"  AUC: {results[name]['auc_mean']:.4f} +/- {results[name]['auc_std']:.4f}")
    print(f"  Recall: {results[name]['recall_mean']:.4f}, Precision: {results[name]['precision_mean']:.4f}")
    print(f"  F1: {results[name]['f1_mean']:.4f}")
    print(f"  Time: {elapsed:.1f}s")

# Summary table
print("\n" + "=" * 85)
print("BASELINE MODEL COMPARISON (5-Fold Stratified CV)")
print("=" * 85)
print(f"{'Model':<25} {'AUC':>14} {'Recall':>14} {'Precision':>14} {'F1':>14} {'Time':>8}")
print("-" * 85)
for name, r in results.items():
    print(f"{name:<25} "
          f"{r['auc_mean']:.4f}+/-{r['auc_std']:.3f} "
          f"{r['recall_mean']:.4f}+/-{r['recall_std']:.3f} "
          f"{r['precision_mean']:.4f}+/-{r['precision_std']:.3f} "
          f"{r['f1_mean']:.4f}+/-{r['f1_std']:.3f} "
          f"{r['time']:>6.1f}s")
print("-" * 85)
print("Success targets: AUC > 0.65, Recall > 0.50")

All four models exceeded both success targets. Logistic Regression led with AUC 0.6982 (+/-0.011) and recall 0.609, followed closely by Random Forest at AUC 0.6961 (+/-0.008) and recall 0.574. XGBoost (AUC 0.6788) and SVM (AUC 0.6754) also cleared the 0.65 threshold but trailed the top two. LR outperforming XGBoost at baseline is a notable surprise given that published benchmarks for this dataset used gradient boosting -- XGBoost's defaults may not be optimal, and Phase 5 tuning may change the ranking. SVM took approximately 45 minutes for 5-fold CV (vs. 6 seconds for Random Forest), delivering the weakest AUC of the group -- its computational cost rules it out for further tuning. LR's 60.9% recall means it catches roughly 3 out of every 5 actual readmissions at the default 0.50 threshold; threshold optimization in Phase 6 can push this higher at the cost of precision.

### 4.3 Baseline Comparison Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

model_names = list(results.keys())
model_colors = [COLORS['steel'], COLORS['teal'], COLORS['orange'], COLORS['gold']]
x_pos = range(len(model_names))

# Top-left: ROC-AUC
ax = axes[0, 0]
means = [results[m]['auc_mean'] for m in model_names]
stds = [results[m]['auc_std'] for m in model_names]
bars = ax.bar(x_pos, means, yerr=stds, color=model_colors, capsize=5,
              edgecolor=COLORS['primary'], linewidth=0.8)
ax.axhline(y=0.65, color=COLORS['red'], linestyle='--', linewidth=1.5, label='Target (0.65)')
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('ROC-AUC by Model', fontweight='bold', fontsize=13)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0.5, 0.75)

# Top-right: Recall
ax = axes[0, 1]
means = [results[m]['recall_mean'] for m in model_names]
stds = [results[m]['recall_std'] for m in model_names]
ax.bar(x_pos, means, yerr=stds, color=model_colors, capsize=5,
       edgecolor=COLORS['primary'], linewidth=0.8)
ax.axhline(y=0.50, color=COLORS['red'], linestyle='--', linewidth=1.5, label='Target (0.50)')
ax.set_ylabel('Recall', fontsize=12)
ax.set_title('Recall by Model', fontweight='bold', fontsize=13)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0.0, 1.0)

# Bottom-left: F1
ax = axes[1, 0]
means = [results[m]['f1_mean'] for m in model_names]
stds = [results[m]['f1_std'] for m in model_names]
ax.bar(x_pos, means, yerr=stds, color=model_colors, capsize=5,
       edgecolor=COLORS['primary'], linewidth=0.8)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('F1-Score by Model', fontweight='bold', fontsize=13)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0.0, 0.4)

# Bottom-right: Training Time
ax = axes[1, 1]
times = [results[m]['time'] for m in model_names]
ax.bar(x_pos, times, color=model_colors, edgecolor=COLORS['primary'], linewidth=0.8)
ax.set_ylabel('Training Time (seconds)', fontsize=12)
ax.set_title('Training Time (5-Fold CV)', fontweight='bold', fontsize=13)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(model_names, rotation=15, ha='right')
# Add time labels on bars
for i, t in enumerate(times):
    ax.text(i, t + max(times) * 0.02, f'{t:.1f}s', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Baseline Model Comparison (5-Fold Stratified CV)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../figures/baseline_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The AUC panel confirms all four models clear the 0.65 target (dashed line), with LR and RF clustered near 0.70 and XGBoost/SVM slightly below. The Recall panel shows LR leading at 0.609 -- all four exceed the 0.50 target, though XGBoost and SVM sit just above the line. F1 scores are uniformly low (~0.21) across all models, reflecting the precision-recall tradeoff inherent in the 8:1 class imbalance. The training time panel makes SVM's cost unmistakable: 2,708 seconds (45 minutes) versus 6 seconds for XGBoost and Random Forest, with no compensating performance advantage.

### 4.4 Class Imbalance Approach Comparison

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import clone

# Identify best model by AUC
best_model_name = max(results, key=lambda k: results[k]['auc_mean'])
print(f"Testing imbalance strategies on: {best_model_name}")

# Get the classifier from the best model
best_classifier = models[best_model_name].named_steps['classifier']

# Strategy 1: Already computed (class_weight='balanced' or scale_pos_weight)
strategy_1_results = results[best_model_name]

# Strategy 2: SMOTE -- must use imblearn Pipeline
smote_classifier = clone(best_classifier)
smote_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', smote_classifier)
])

print("\nRunning SMOTE strategy...")
smote_cv = cross_validate(smote_pipeline, X_train, y_train,
                          cv=cv, scoring=scoring, n_jobs=-1)

# Strategy 3: No imbalance handling
no_balance_classifier = clone(best_classifier)
if hasattr(no_balance_classifier, 'class_weight'):
    no_balance_classifier.set_params(class_weight=None)
elif hasattr(no_balance_classifier, 'scale_pos_weight'):
    no_balance_classifier.set_params(scale_pos_weight=1)

no_balance_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', no_balance_classifier)
])

print("Running no-imbalance-handling strategy...")
no_balance_cv = cross_validate(no_balance_pipeline, X_train, y_train,
                               cv=cv, scoring=scoring, n_jobs=-1)

print("All three imbalance strategies evaluated.")

In [ ]:
# Comparison table
strategies = {
    'class_weight=balanced': {
        'auc': strategy_1_results['auc_mean'],
        'recall': strategy_1_results['recall_mean'],
        'precision': strategy_1_results['precision_mean'],
        'f1': strategy_1_results['f1_mean'],
    },
    'SMOTE': {
        'auc': smote_cv['test_roc_auc'].mean(),
        'recall': smote_cv['test_recall'].mean(),
        'precision': smote_cv['test_precision'].mean(),
        'f1': smote_cv['test_f1'].mean(),
    },
    'No handling': {
        'auc': no_balance_cv['test_roc_auc'].mean(),
        'recall': no_balance_cv['test_recall'].mean(),
        'precision': no_balance_cv['test_precision'].mean(),
        'f1': no_balance_cv['test_f1'].mean(),
    },
}

print(f"Imbalance Strategy Comparison -- {best_model_name}")
print("=" * 65)
print(f"{'Strategy':<25} {'AUC':>10} {'Recall':>10} {'Precision':>10} {'F1':>10}")
print("-" * 65)
for name, m in strategies.items():
    print(f"{name:<25} {m['auc']:>10.4f} {m['recall']:>10.4f} {m['precision']:>10.4f} {m['f1']:>10.4f}")
print("-" * 65)

# Grouped bar chart -- AUC and Recall side by side for each strategy
fig, ax = plt.subplots(figsize=(12, 5))

strategy_names = list(strategies.keys())
x = np.arange(len(strategy_names))
width = 0.3

auc_vals = [strategies[s]['auc'] for s in strategy_names]
recall_vals = [strategies[s]['recall'] for s in strategy_names]

bars1 = ax.bar(x - width/2, auc_vals, width, label='ROC-AUC',
               color=COLORS['steel'], edgecolor=COLORS['primary'], linewidth=0.8)
bars2 = ax.bar(x + width/2, recall_vals, width, label='Recall',
               color=COLORS['orange'], edgecolor=COLORS['primary'], linewidth=0.8)

ax.axhline(y=0.65, color=COLORS['steel'], linestyle='--', linewidth=1, alpha=0.6, label='AUC target (0.65)')
ax.axhline(y=0.50, color=COLORS['orange'], linestyle='--', linewidth=1, alpha=0.6, label='Recall target (0.50)')

ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Class Imbalance Strategy Comparison -- {best_model_name}',
             fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(strategy_names, fontsize=11)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.0)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/imbalance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The `class_weight='balanced'` strategy (AUC 0.6982, recall 0.609) slightly outperformed SMOTE (AUC 0.6831, recall 0.598), confirming that both effectively address the ~8:1 class imbalance. The "no handling" result is the most striking: recall collapsed to 0.008, meaning the model caught fewer than 1 in 100 actual readmissions. Without class balancing, the classifier predicts "not readmitted" for virtually every patient -- technically high accuracy but clinically useless. This is the strongest argument for cost-sensitive learning: `class_weight='balanced'` delivers comparable or better performance to SMOTE without the added complexity and memory overhead of synthetic oversampling, and it avoids the risk of SMOTE generating unrealistic synthetic examples in a sparse, high-dimensional feature space.

### 4.5 Per-Fold Performance Stability

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

fold_data = []
for name, r in results.items():
    for fold_auc in r['cv_results']['test_roc_auc']:
        fold_data.append({'Model': name, 'AUC': fold_auc})

fold_df = pd.DataFrame(fold_data)

# Box plot with individual points overlaid
model_colors_list = [COLORS['steel'], COLORS['teal'], COLORS['orange'], COLORS['gold']]
model_list = list(results.keys())

bp = ax.boxplot(
    [fold_df[fold_df['Model'] == m]['AUC'].values for m in model_list],
    labels=model_list,
    patch_artist=True, widths=0.5
)

for patch, color in zip(bp['boxes'], model_colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor(COLORS['primary'])

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color(COLORS['primary'])
        if element == 'medians':
            item.set_linewidth(2)

# Overlay individual fold points with jitter
np.random.seed(42)
for i, name in enumerate(model_list):
    fold_aucs = results[name]['cv_results']['test_roc_auc']
    jitter = np.random.normal(0, 0.04, size=len(fold_aucs))
    ax.scatter(np.ones(len(fold_aucs)) * (i + 1) + jitter, fold_aucs,
               color=COLORS['primary'], alpha=0.7, s=50, zorder=5, edgecolors='white', linewidth=0.5)

ax.axhline(y=0.65, color=COLORS['red'], linestyle='--', linewidth=1.5, label='Target (0.65)')
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('Per-Fold AUC Stability (5-Fold Stratified CV)', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../figures/cv_fold_stability.png', dpi=150, bbox_inches='tight')
plt.show()

Logistic Regression shows the widest fold-level AUC spread (+/-0.011) but still has the highest median, indicating its lead is genuine rather than driven by a single lucky fold. Random Forest is the most stable (+/-0.008) with a tight interquartile range, making it a reliable runner-up. XGBoost (+/-0.006) and SVM (+/-0.008) are both stable but clustered at a lower AUC level. All four models remain above the 0.65 target across every fold, confirming that the baseline results are robust to the particular train/validation split.

### 4.6 Baseline Training on Full Training Set

In [ ]:
import joblib

# Identify top 2 models
sorted_models = sorted(results, key=lambda k: results[k]['auc_mean'], reverse=True)
top_2 = sorted_models[:2]

for name in top_2:
    models[name].fit(X_train, y_train)
    joblib.dump(models[name], f'../models/baseline_{name.lower().replace(" ", "_")}.joblib')
    print(f"Fitted and saved: {name} (CV AUC: {results[name]['auc_mean']:.4f})")

best_model_name = top_2[0]
print(f"\nBest baseline: {best_model_name}")
print(f"Runner-up:     {top_2[1]}")
print(f"NOTE: Test set remains locked until Phase 6.")

The top two baseline models have been fitted on the full training set and saved to `./models/`. These are the starting points for Phase 5 hyperparameter tuning. They have **not** been evaluated on the held-out test set -- that evaluation is reserved for Phase 6 to prevent data leakage and ensure an unbiased estimate of generalization performance.

### Phase 4 Complete

**Baseline results:**
- [x] Utility functions defined (evaluate_model, predict_at_threshold, compute_business_impact)
- [x] 4 models trained with default hyperparameters
- [x] 5-fold stratified CV on all 4 models (AUC, Recall, Precision, F1)
- [x] Best model identified: Logistic Regression (AUC: 0.6982)
- [x] Class imbalance comparison: balanced weights vs. SMOTE vs. no handling
- [x] Per-fold stability analysis
- [x] Top 2 baselines saved to ./models/ (Logistic Regression, Random Forest)
- [x] Test set remains LOCKED

**Success metric check:**
- AUC > 0.65 target: MET (all four models -- LR 0.698, RF 0.696, XGB 0.679, SVM 0.675)
- Recall > 0.50 target: MET (all four models -- LR 0.609, RF 0.574, XGB 0.509, SVM 0.504)

**Next: Phase 5 -- Hyperparameter Tuning**

### Save Phase 4 Artifacts

In [ ]:
# Save Phase 4 artifacts for Phase 5

# Save CV results
with open(f'{ARTIFACTS_DIR}/phase4_cv_results.pkl', 'wb') as f:
    pickle.dump(results, f)

# Save all pipeline objects
for name, pipeline in models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(pipeline, f'{ARTIFACTS_DIR}/phase4_baseline_{safe_name}.joblib')
    print(f'  Saved: phase4_baseline_{safe_name}.joblib')

# Save Phase 4 metadata
phase4_meta = {
    'results': {name: {
        'auc_mean': float(results[name]['auc_mean']),
        'auc_std': float(results[name]['auc_std']),
        'recall_mean': float(results[name]['recall_mean']),
        'precision_mean': float(results[name]['precision_mean']),
        'f1_mean': float(results[name]['f1_mean']),
        'time': float(results[name]['time']),
    } for name in results},
    'best_model_name': best_model_name,
    'model_names': list(models.keys()),
}
with open(f'{ARTIFACTS_DIR}/phase4_meta.pkl', 'wb') as f:
    pickle.dump(phase4_meta, f)

print(f'\nPhase 4 artifacts saved')
print(f'  phase4_cv_results.pkl')
print(f'  phase4_meta.pkl')
print(f'  phase4_baseline_*.joblib x {len(models)}')